# Lung Sound Classificaation using Machine Learning

## Objective
Build a machine learning model that classifies lung sound into different medical conditions using audio recordings

## Dataset
HLS_CMDS: Heart and Lung sounds Dataset (UCI Repository)

We use:
- LS.csv (labels + metadata)
- WAV files (audio signals)

Each Lung Sound ID corresponds to a .wav file

In [1]:
import pandas as pd
import numpy as np 
import os 
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [2]:
# Load the dataset
df = pd.read_csv("../dataset/LS.csv")

print(df.head())
print(df["Lung Sound Type"].value_counts())

  Gender Lung Sound Type Location Lung Sound ID
0      M          Normal      RUA       M_N_RUA
1      F          Normal      LUA       F_N_LUA
2      F          Normal      RMA       F_N_RMA
3      F          Normal      LMA       F_N_LMA
4      M          Normal      RLA       M_N_RLA
Lung Sound Type
Normal             12
Pleural Rub         9
Coarse Crackles     9
Rhonchi             8
Wheezing            7
Fine Crackles       5
Name: count, dtype: int64


In [3]:
# clean data
counts = df["Lung Sound Type"].value_counts()

valid_classes = counts[counts >= 5].index
df = df[df["Lung Sound Type"].isin(valid_classes)]

print("Final classes: \n", df["Lung Sound Type"].value_counts())

Final classes: 
 Lung Sound Type
Normal             12
Pleural Rub         9
Coarse Crackles     9
Rhonchi             8
Wheezing            7
Fine Crackles       5
Name: count, dtype: int64


In [4]:
# Map audio filepaths
df["Lung Sound ID"] = df["Lung Sound ID"].astype(str).str.strip()

base_path= "../dataset/LS"

df["filepath"] = df["Lung Sound ID"].apply(
    lambda x: os.path.join(base_path, x+ ".wav")
)

In [5]:
# feature extraction
X = []
y = []

valid_count = 0

for _, row in df.iterrows():
    path = row["filepath"]
    label = row["Lung Sound Type"]

    if not os.path.exists(path):
        print("Missing:", path)
        continue
    
    try:
        audio, sr = librosa.load(path, sr = 22050)
        mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20), axis=1)

        zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio))
        rms = np.mean(librosa.feature.rms(y=audio))
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
        bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))

        features = np.hstack([mfcc, zcr, rms, centroid, bandwidth])

        X.append(features)
        y.append(label)

        valid_count += 1

    except Exception as e:
        print("Error:", path, e)

print("Valid samples loaded:", valid_count)

    

Valid samples loaded: 50


In [6]:
# converting to ml format
X = np.array(X)
y = np.array(y)

print("X shape: ", X.shape)
print("y shape: ", y.shape)


X shape:  (50, 24)
y shape:  (50,)
